In [4]:
import numpy as np
import pandas as pd
import scipy.stats
import gudhi as gd
import networkx as nx
import random
from tqdm import tqdm
import os

In [6]:
# --- Persistent Homology Computation ---
def compute_persistent_homology(graph, filtration):
    """
    Compute the persistent homology of a graph with a given filtration.

    :param graph: A networkx graph.
    :param filtration: A list of node filtration values, corresponding to each node in the graph.
    :return: Persistent homology of dimension one.
    """
    # Create simplex tree
    st = gd.SimplexTree()

    # Add vertices with filtration values
    for i, node in enumerate(graph.nodes):
        st.insert([node], filtration=filtration[i])

    # Add edges
    for edge in graph.edges:
        st.insert(list(edge), filtration=max(
            filtration[list(graph.nodes()).index(edge[0])],
            filtration[list(graph.nodes()).index(edge[1])]
        ))

    # Expand to clique complex
    st.extend_filtration()

    # Remove null cycles
    persistence = st.extended_persistence(min_persistence=1e-5)
    tmp = []
    tmp.extend(persistence[0])
    tmp.extend(persistence[1])
    tmp.extend(persistence[2])
    tmp.extend(persistence[3])
    persistence = tmp

    # Ensure birth <= death
    tmp = []
    for gen in persistence:
        tmp.append((gen[0], (min(gen[1][0], gen[1][1]), max(gen[1][0], gen[1][1]))))

    persistence = tmp

    # Filter for dimension one homology
    dim1_res = []
    for gen in persistence:
        if gen[0] == 1:
            if gen[1][0] != gen[1][1]:
                dim1_res.append(gen)
    return dim1_res

In [12]:
# --- Add EPH and Correlation ---
def add_eph_and_corr_to_csv(csv_path, graph):
    """
    Add EPH and correlation (corr) columns to the given CSV file based on the graph.

    :param csv_path: Path to the CSV file.
    :param graph: The corresponding graph (networkx.Graph).
    """
    sims = pd.read_csv(csv_path)

    # Remove extra columns based on expected CSV format
    order_columns = [col for col in sims.columns if str(col).isdigit() or (str(col).lstrip('-').isdigit())]
    order = sims[order_columns].copy(deep=True)
    order_prl = order.copy(deep=True)
    order = order.apply(lambda row: row.fillna(row.max() + 1), axis=1)

    EPH = []
    corr = []

    for row_index in tqdm(range(len(order))):
        # Define filtration
        filt = list(-order.iloc[row_index].values)

        # Compute EPH
        persistent_homology = compute_persistent_homology(graph=graph, filtration=filt)

        # Lifetime of generators
        life_set = []
        for gen in persistent_homology:
            life_set.append(gen[1][1] - gen[1][0])
        EPH.append(np.nanmean(life_set))

        # Compute PRL (Spearman correlation)
        deg_list = list(dict(graph.degree).values())
        order_curr = list(order_prl.iloc[row_index].values)

        try:
            rho, _ = scipy.stats.spearmanr(deg_list, order_curr, nan_policy='omit')
        except Exception as e:
            print(f'PRL method could not find rho for row {row_index}: {e}')
            rho = 0

        corr.append(rho)

    sims['EPH'] = EPH
    sims['corr'] = corr

    output_csv = csv_path.replace('.csv', '_EPH.csv')
    sims.to_csv(output_csv, index=False)
    print(f"Updated CSV saved to {output_csv}")


In [16]:
# --- Main Function ---
def main():
    # Network and CSV details
    network_list = ['conf', 'email_eu_modified', 'hospital', 'school', 'work']
    simple_betas = [0.02, 0.03, 0.04]
    complex_thetas = [2, 3, 4, 5, 6, 7]
    complex_probs = [-1, 0.02, 0.03, 0.04]

    results_dir = '../results/unweighted/'
    network_dir = '../networks/unweighted/'

    for network_name in network_list:
        # Load and preprocess graph
        graph_path = os.path.join(network_dir, f'G_unweighted_{network_name}.graphml')
        G = nx.read_graphml(graph_path)

        mapping = {node: int(float(node)) for node in G.nodes()}
        G = nx.relabel_nodes(G, mapping)

        # Process simple contagion CSVs
        for beta in simple_betas:
            csv_path = os.path.join(results_dir, f'unweighted_simple_{network_name}_beta_{beta}.csv')
            if os.path.exists(csv_path):
                add_eph_and_corr_to_csv(csv_path, G)

        # Process complex contagion CSVs
        for theta in complex_thetas:
            for prob_complex in complex_probs:
                csv_path = os.path.join(results_dir, f'unweighted_complex_{network_name}_theta_{theta}_prob_{prob_complex}.csv')
                if os.path.exists(csv_path):
                    add_eph_and_corr_to_csv(csv_path, G)


In [18]:

if __name__ == "__main__":
    main()

columns =  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156',

100%|█████████████████████████████████████████| 100/100 [00:05<00:00, 17.35it/s]


Updated CSV saved to ../results/unweighted/unweighted_simple_conf_beta_0.02_EPH.csv
columns =  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144',

100%|█████████████████████████████████████████| 100/100 [00:05<00:00, 17.43it/s]


Updated CSV saved to ../results/unweighted/unweighted_simple_conf_beta_0.03_EPH.csv
columns =  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144',

...

Updated CSV saved to ../results/unweighted/unweighted_complex_conf_theta_2_prob_0.02_EPH.csv
columns =  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143

100%|█████████████████████████████████████████| 100/100 [00:05<00:00, 18.60it/s]


Updated CSV saved to ../results/unweighted/unweighted_complex_conf_theta_2_prob_0.03_EPH.csv
columns =  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143

 85%|███████████████████████████████████▋      | 85/100 [00:04<00:00, 18.68it/s]


KeyboardInterrupt: 